In [ ]:

# ============================================================
# CYBERGUARD AI
# PHASE 6 — COMPLETE MODEL TRAINING & EVALUATION
# Dataset: UNSW-NB15
# Target: 0 = Benign, 1 = Malicious
# ============================================================

# 1. IMPORT LIBRARIES

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
)


# ============================================================
# 2. FIND THE CORRECT PROJECT FOLDER
# ============================================================

# Search the current folder and its parents.
# This works whether Jupyter starts in the repository
# root or inside the notebooks folder.

current_folder = Path.cwd().resolve()
project_root = None

for folder in [current_folder, *current_folder.parents]:

    candidate = (
        folder
        / "Data"
        / "Raw data"
        / "UNSW_NB15_training-set.csv"
    )

    if candidate.is_file():
        project_root = folder
        break

if project_root is None:
    raise FileNotFoundError(
        "Could not locate CyberGuard AI. "
        "Check that the Data/Raw data folder exists."
    )

data_dir = project_root / "Data" / "Raw data"
docs_dir = project_root / "docs"
figures_dir = project_root / "assets" / "model_evaluation"

docs_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

print("=== PROJECT CONFIGURATION ===")
print("Project root:", project_root)
print("Data folder:", data_dir)


# ============================================================
# 3. LOAD TRAINING AND TEST DATASETS
# ============================================================

train_path = data_dir / "UNSW_NB15_training-set.csv"
test_path = data_dir / "UNSW_NB15_testing-set.csv"

if not test_path.is_file():
    raise FileNotFoundError(
        f"Test Dataset not found: {test_path}"
    )

df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)

print("\n=== DATASET LOADING ===")
print("Training:", df_train.shape)
print("Testing:", df_test.shape)

assert df_train.shape[1] == 45
assert df_test.shape[1] == 45

assert df_train["label"].isin([0, 1]).all()
assert df_test["label"].isin([0, 1]).all()


# ============================================================
# 4. FEATURE SELECTION
# ============================================================

# Exclude identifier and attack category.
# The label is our target, not an input feature.

excluded_columns = ["id", "attack_cat", "label"]

feature_cols = [
    col
    for col in df_train.columns
    if col not in excluded_columns
]

assert len(feature_cols) == 42
assert set(feature_cols).issubset(df_test.columns)

df_clean = df_train[
    feature_cols + ["label"]
].copy()

print("\n=== FEATURE SELECTION ===")
print("Features:", len(feature_cols))
print("Working dataset:", df_clean.shape)


# ============================================================
# 5. TRAINING / VALIDATION SPLIT
# ============================================================

# Group identical input feature combinations.
# This reduces duplicate-feature overlap between
# model training and validation.

groups = pd.util.hash_pandas_object(
    df_clean[feature_cols],
    index=False
)

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(df_clean, groups=groups)
)

df_fit = df_clean.iloc[train_idx].copy()
df_val = df_clean.iloc[val_idx].copy()

shared_groups = (
    set(groups.iloc[train_idx])
    & set(groups.iloc[val_idx])
)

assert len(shared_groups) == 0

X_train = df_fit[feature_cols].copy()
y_train = df_fit["label"].copy()

X_val = df_val[feature_cols].copy()
y_val = df_val["label"].copy()

print("\n=== DATA SPLIT ===")
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("Shared feature groups:", len(shared_groups))


# ============================================================
# 6. DEFINE PREPROCESSING
# ============================================================

numerical_features = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

assert len(numerical_features) == 39
assert len(categorical_features) == 3

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            "passthrough",
            numerical_features
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            ),
            categorical_features
        )
    ],
    sparse_threshold=1.0
)

print("\n=== PREPROCESSING ===")
print("Numerical features:", len(numerical_features))
print("Categorical features:", categorical_features)


# ============================================================
# 7. EVALUATION FUNCTIONS
# ============================================================

def calculate_metrics(y_true, predictions, probabilities):

    return {
        "Accuracy": accuracy_score(
            y_true, predictions
        ),
        "Balanced Accuracy": balanced_accuracy_score(
            y_true, predictions
        ),
        "Precision": precision_score(
            y_true, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_true, predictions, zero_division=0
        ),
        "F1-score": f1_score(
            y_true, predictions, zero_division=0
        ),
        "ROC-AUC": roc_auc_score(
            y_true, probabilities
        )
    }


def display_metrics(title, metrics):

    print(f"\n=== {title} ===")

    for metric, value in metrics.items():
        print(f"{metric}: {value:.4f}")


def report_table_row(name, metrics):

    values = " | ".join(
        f"{value:.4f}"
        for value in metrics.values()
    )

    return f"| {name} | {values} |"


# ============================================================
# 8. TRAIN BASELINE MODEL
# ============================================================

print("\n=== BASELINE MODEL TRAINING ===", flush=True)

# DummyClassifier does not need real features
# because it predicts the most frequent class.

baseline = DummyClassifier(
    strategy="most_frequent"
)

baseline_train_input = np.zeros(
    (len(y_train), 1)
)

baseline_val_input = np.zeros(
    (len(y_val), 1)
)

baseline.fit(
    baseline_train_input,
    y_train
)

baseline_predictions = baseline.predict(
    baseline_val_input
)

baseline_probabilities = baseline.predict_proba(
    baseline_val_input
)[:, 1]

baseline_metrics = calculate_metrics(
    y_val,
    baseline_predictions,
    baseline_probabilities
)

display_metrics(
    "BASELINE VALIDATION",
    baseline_metrics
)


# ============================================================
# 9. TRAIN RANDOM FOREST
# ============================================================

print("\n=== RANDOM FOREST TRAINING ===", flush=True)
print("Training 80 Decision Trees...", flush=True)
print("This may take several minutes.", flush=True)

# All preprocessing takes place inside the Pipeline.
# The encoder learns categories only from X_train.

model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=80,
                max_depth=18,
                min_samples_leaf=2,
                class_weight="balanced_subsample",
                random_state=42,
                n_jobs=4
            )
        )
    ]
)

model.fit(X_train, y_train)

print("Random Forest training completed!", flush=True)


# ============================================================
# 10. VALIDATION PREDICTIONS
# ============================================================

val_predictions = model.predict(X_val)

val_probabilities = model.predict_proba(
    X_val
)[:, 1]

validation_metrics = calculate_metrics(
    y_val,
    val_predictions,
    val_probabilities
)

display_metrics(
    "RANDOM FOREST VALIDATION",
    validation_metrics
)

print("\n=== VALIDATION CLASSIFICATION REPORT ===")

print(
    classification_report(
        y_val,
        val_predictions,
        labels=[0, 1],
        target_names=["Benign", "Malicious"],
        zero_division=0
    )
)


# ============================================================
# 11. BASELINE VS RANDOM FOREST
# ============================================================

comparison = pd.DataFrame([
    {
        "Model": "DummyClassifier",
        **baseline_metrics
    },
    {
        "Model": "Random Forest",
        **validation_metrics
    }
])

print("\n=== MODEL COMPARISON ===")

print(
    comparison.to_string(
        index=False,
        float_format=lambda value: f"{value:.4f}"
    )
)


# ============================================================
# 12. VALIDATION CONFUSION MATRIX
# ============================================================

fig, ax = plt.subplots(figsize=(7, 5))

ConfusionMatrixDisplay.from_predictions(
    y_val,
    val_predictions,
    labels=[0, 1],
    display_labels=["Benign", "Malicious"],
    values_format="d",
    ax=ax
)

ax.set_title(
    "CyberGuard AI — Validation Confusion Matrix"
)

fig.tight_layout()

confusion_path = (
    figures_dir / "validation_confusion_matrix.png"
)

fig.savefig(
    confusion_path,
    dpi=150,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)


# ============================================================
# 13. VALIDATION ROC CURVE
# ============================================================

fpr, tpr, _ = roc_curve(
    y_val,
    val_probabilities
)

fig, ax = plt.subplots(figsize=(7, 5))

ax.plot(
    fpr,
    tpr,
    label=(
        "Random Forest "
        f"(AUC = {validation_metrics['ROC-AUC']:.3f})"
    )
)

ax.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random baseline"
)

ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("CyberGuard AI — Validation ROC Curve")
ax.legend()

fig.tight_layout()

roc_path = figures_dir / "validation_roc_curve.png"

fig.savefig(
    roc_path,
    dpi=150,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)


# ============================================================
# 14. FEATURE IMPORTANCE
# ============================================================

fitted_preprocessor = (
    model.named_steps["preprocessor"]
)

fitted_classifier = (
    model.named_steps["classifier"]
)

encoded_names = (
    fitted_preprocessor.get_feature_names_out()
)

importances = pd.Series(
    fitted_classifier.feature_importances_,
    index=encoded_names
).sort_values(ascending=False)

top_features = importances.head(15)

print("\n=== TOP 15 FEATURE IMPORTANCES ===")
print(top_features.to_string())

fig, ax = plt.subplots(figsize=(9, 6))

top_features.sort_values().plot(
    kind="barh",
    ax=ax
)

ax.set_xlabel("Feature Importance")
ax.set_title("CyberGuard AI — Top 15 Features")

fig.tight_layout()

importance_path = (
    figures_dir / "feature_importance.png"
)

fig.savefig(
    importance_path,
    dpi=150,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

# These are model-based importance scores,
# not proof that a feature causes an attack.


# ============================================================
# 15. OFFICIAL TEST EVALUATION
# ============================================================

# The model configuration is fixed.
# Do not change its parameters based on Test scores.

print("\n=== OFFICIAL TEST EVALUATION ===", flush=True)

X_test = df_test[feature_cols].copy()
y_test = df_test["label"].copy()

test_predictions = model.predict(X_test)

test_probabilities = model.predict_proba(
    X_test
)[:, 1]

test_metrics = calculate_metrics(
    y_test,
    test_predictions,
    test_probabilities
)

display_metrics(
    "RANDOM FOREST OFFICIAL TEST",
    test_metrics
)

print("\n=== TEST CLASSIFICATION REPORT ===")

print(
    classification_report(
        y_test,
        test_predictions,
        labels=[0, 1],
        target_names=["Benign", "Malicious"],
        zero_division=0
    )
)


# ============================================================
# 16. TRAIN / TEST OVERLAP AUDIT
# ============================================================

# Audit A:
# Compare Test against the full official Training CSV.

full_train_hashes = pd.util.hash_pandas_object(
    df_train[feature_cols],
    index=False
)

# Audit B:
# Compare Test against records actually used to fit
# the Random Forest (the development Training subset).

fit_hashes = pd.util.hash_pandas_object(
    X_train,
    index=False
)

test_hashes = pd.util.hash_pandas_object(
    X_test,
    index=False
)

overlap_official = test_hashes.isin(
    set(full_train_hashes)
).to_numpy()

overlap_model_fit = test_hashes.isin(
    set(fit_hashes)
).to_numpy()

official_overlap_count = int(
    overlap_official.sum()
)

model_overlap_count = int(
    overlap_model_fit.sum()
)

print("\n=== TRAIN/TEST OVERLAP ===")

print(
    "Matches against full official Training:",
    official_overlap_count
)

print(
    "Matches against model fitting subset:",
    model_overlap_count
)

print(
    "Test records without fitting-set matches:",
    int((~overlap_model_fit).sum())
)

# Phase 5 previously confirmed exact feature matches
# for the official split, including 8,541 matching
# test records. Hashes are used here for efficient
# reproduction of the same overlap audit.


# ============================================================
# 17. NON-OVERLAPPING TEST EVALUATION
# ============================================================

# Supplementary analysis:
# Evaluate Test records that do not match the
# combinations actually used to fit the model.

nonoverlap_mask = ~overlap_model_fit

y_test_nonoverlap = y_test.to_numpy()[
    nonoverlap_mask
]

pred_nonoverlap = test_predictions[
    nonoverlap_mask
]

prob_nonoverlap = test_probabilities[
    nonoverlap_mask
]

assert len(y_test_nonoverlap) > 0
assert len(np.unique(y_test_nonoverlap)) == 2

nonoverlap_metrics = calculate_metrics(
    y_test_nonoverlap,
    pred_nonoverlap,
    prob_nonoverlap
)

display_metrics(
    "TEST — NO MODEL-FIT FEATURE OVERLAP",
    nonoverlap_metrics
)


# ============================================================
# 18. ERROR ANALYSIS
# ============================================================

tn, fp, fn, tp = confusion_matrix(
    y_test,
    test_predictions,
    labels=[0, 1]
).ravel()

print("\n=== OFFICIAL TEST ERROR ANALYSIS ===")

print("True Negatives:", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives:", tp)

print("\nFalse Positive = Benign classified as Malicious")
print("False Negative = Malicious classified as Benign")


# ============================================================
# 19. GENERATE MODEL EVALUATION REPORT
# ============================================================

metrics_table = "\n".join([
    "| Evaluation | Accuracy | Balanced Accuracy | Precision | Recall | F1-score | ROC-AUC |",
    "|---|---:|---:|---:|---:|---:|---:|",
    report_table_row(
        "Baseline Validation",
        baseline_metrics
    ),
    report_table_row(
        "Random Forest Validation",
        validation_metrics
    ),
    report_table_row(
        "Random Forest Official Test",
        test_metrics
    ),
    report_table_row(
        "Test Without Model-Fit Overlap",
        nonoverlap_metrics
    ),
])

importance_rows = "\n".join(
    f"| {name} | {value:.4f} |"
    for name, value in importances.head(10).items()
)

report = f"""# CyberGuard AI — Model Training & Evaluation

## 1. Objective

Train and evaluate a supervised Machine Learning
model for binary network traffic classification.

- Benign: 0
- Malicious: 1

This is an educational offline research project,
not a production cybersecurity detection system.

## 2. Dataset

- Dataset: UNSW-NB15
- Official Training Records: {len(df_train):,}
- Official Test Records: {len(df_test):,}
- Input Features: {len(feature_cols)}
- Numerical Features: {len(numerical_features)}
- Categorical Features: {len(categorical_features)}
- Encoded Features: {len(encoded_names)}

Excluded: `id` and `attack_cat`.

## 3. Development Split

- Method: GroupShuffleSplit
- Random State: 42
- Model Training Records: {len(X_train):,}
- Validation Records: {len(X_val):,}
- Shared Feature Groups: {len(shared_groups)}

The Random Forest was fitted on the development
Training subset, rather than all records in the
official Training CSV.

## 4. Preprocessing

Numerical Features:
- Passthrough

Categorical Features:
- OneHotEncoder
- Unknown categories ignored

Preprocessing was fitted only on X_train.

## 5. Models

### Baseline

DummyClassifier:
- Strategy: most_frequent

### Random Forest

RandomForestClassifier:
- n_estimators: 80
- max_depth: 18
- min_samples_leaf: 2
- class_weight: balanced_subsample
- random_state: 42
- n_jobs: 4

These were initial fixed settings.
No hyperparameter search was performed.

## 6. Evaluation Metrics

Positive class: Malicious (1).

{metrics_table}

Validation was used to assess the model during
development.

The fitted Random Forest was evaluated on the
official Test Dataset without retraining.

The official Test results must not be used to
select new hyperparameters.

## 7. Official Test Confusion Matrix

- True Negatives: {tn:,}
- False Positives: {fp:,}
- False Negatives: {fn:,}
- True Positives: {tp:,}

## 8. Train/Test Feature Overlap

- Matches against the full official Training set: {official_overlap_count:,}
- Matches against the model fitting subset: {model_overlap_count:,}
- Test records without fitting-set feature matches: {int(nonoverlap_mask.sum()):,}

The full official Training/Test overlap was
previously verified with exact feature comparisons
during Phase 5.

Identical features do not necessarily mean
identical real-world network connections.

The non-overlapping Test evaluation is supplementary.

## 9. Top 10 Feature Importances

| Feature | Importance |
|---|---:|
{importance_rows}

These are impurity-based model importance scores,
not causal explanations.

## 10. Visualizations

![Validation Confusion Matrix](../assets/model_evaluation/validation_confusion_matrix.png)

![Validation ROC Curve](../assets/model_evaluation/validation_roc_curve.png)

![Feature Importance](../assets/model_evaluation/feature_importance.png)

## 11. Limitations

- Historical network traffic, not live data.
- Original Duplicate Records retained.
- Conflicting Labels not automatically corrected.
- Class imbalance exists.
- Train/Test feature overlap exists.
- Evaluation may not reflect real-world performance.
- The model has not been deployed or externally validated.

## 12. Next Steps

- Review False Positives and False Negatives.
- Improve portfolio documentation.
- Present findings and limitations.
- Consider external validation on another dataset.
"""

report_path = (
    docs_dir / "model-evaluation-report.md"
)

report_path.write_text(
    report,
    encoding="utf-8"
)


# ============================================================
# 20. FINAL SUMMARY
# ============================================================

print("\n=== FILES GENERATED ===")
print(report_path)
print(confusion_path)
print(roc_path)
print(importance_path)

print("\n" + "=" * 55)
print("CYBERGUARD AI — PHASE 6 COMPLETE")
print("=" * 55)

print("Baseline Model: Complete")
print("Random Forest Training: Complete")
print("Validation Evaluation: Complete")
print("Official Test Evaluation: Complete")
print("Feature Overlap Analysis: Complete")
print("Error Analysis: Complete")
print("Visualizations: Complete")
print("Technical Report: Complete")

print(
    "\nVALIDATION F1-SCORE:",
    f"{validation_metrics['F1-score']:.4f}"
)

print(
    "OFFICIAL TEST F1-SCORE:",
    f"{test_metrics['F1-score']:.4f}"
)

print("\nReady for Project Review & Portfolio Preparation.")


In [ ]:

# Verify notebook and generated files
git status --short

# Add Phase 6 files
git add notebooks/03_model_training.ipynb
git add docs/model-evaluation-report.md
git add assets/model_evaluation/

# Review staged files
git diff --cached --stat

# Create commit
git commit -m "Complete Phase 6 model training and evaluation"

# Publish to GitHub
git push origin main
